# Subliminal Mitigate — Evaluation Plots

Reads one or more `results.json` files produced by `evaluate.py` and plots:
- Instruction following quality (mean helpfulness)
- Subliminal effect strength (probe-type breakdown)
- Coding security rate (code_security type only)

Models with `null` entries are silently skipped in every plot.

In [ ]:
# ── Configure here ────────────────────────────────────────────────────────────
# Single file:
results_path = "../outputs/results.json"

# Optional: compare multiple runs side-by-side (leave as [] to use single file)
# Each entry is a dict with 'path' and a display 'label'.
# Example:
#   compare_runs = [
#       {"path": "../outputs/results_kl.json",             "label": "kl"},
#       {"path": "../outputs/results_shared_subspace.json","label": "shared_subspace"},
#   ]
compare_runs = []
# ──────────────────────────────────────────────────────────────────────────────

In [ ]:
import json
import os
import warnings

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

warnings.filterwarnings("ignore")

MODEL_NAMES   = ["pi_A", "pi_B", "pi_baseline", "pi_reg"]
MODEL_COLORS  = {
    "pi_A":        "#4C72B0",
    "pi_B":        "#DD8452",
    "pi_baseline": "#55A868",
    "pi_reg":      "#C44E52",
}
plt.rcParams.update({"figure.dpi": 120, "font.size": 11, "axes.spines.top": False, "axes.spines.right": False})

In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────

def load_run(path, label=None):
    """Load a results JSON. Returns (label, meta, {model: results_or_None})."""
    if not os.path.isfile(path):
        raise FileNotFoundError(f"Results file not found: {path}")
    with open(path) as f:
        raw = json.load(f)
    meta = raw.get("meta", {})
    model_results = {name: raw.get(name) for name in MODEL_NAMES}
    run_label = label or meta.get("subliminal_type", os.path.basename(path))
    return run_label, meta, model_results


if compare_runs:
    runs = [load_run(r["path"], r.get("label")) for r in compare_runs]
else:
    runs = [load_run(results_path)]

# Use first run's meta for display
primary_label, primary_meta, primary_results = runs[0]
subliminal_type = primary_meta.get("subliminal_type", "unknown")

print(f"Subliminal type : {subliminal_type}")
print(f"Base model      : {primary_meta.get('base_model', 'unknown')}")
print(f"Checkpoint dir  : {primary_meta.get('checkpoint_dir', 'unknown')}")
print(f"Timestamp       : {primary_meta.get('timestamp', 'unknown')}")
print(f"Runs loaded     : {len(runs)}")
for run_label, _, model_results in runs:
    available = [m for m, v in model_results.items() if v is not None]
    null_models = [m for m, v in model_results.items() if v is None]
    print(f"  [{run_label}] evaluated: {available}  |  null: {null_models}")

In [ ]:
# ── Plotting helpers ──────────────────────────────────────────────────────────

def _safe_get(model_results, model, *keys):
    """Safely traverse nested dict; return None if any key is missing or value is None."""
    node = model_results.get(model)
    for key in keys:
        if not isinstance(node, dict):
            return None
        node = node.get(key)
    return node


def single_run_bar(ax, model_results, key_path, title, ylabel, fmt=".3f"):
    """
    Bar chart for a single run.  key_path is a tuple of keys to traverse.
    Only models with non-None values are plotted.
    """
    models = [m for m in MODEL_NAMES if _safe_get(model_results, m, *key_path) is not None]
    if not models:
        ax.set_title(f"{title}\n(no data)")
        ax.axis("off")
        return
    values = [_safe_get(model_results, m, *key_path) for m in models]
    colors = [MODEL_COLORS[m] for m in models]
    bars = ax.bar(models, values, color=colors, edgecolor="white", linewidth=0.8)
    ax.bar_label(bars, fmt=f"%{fmt}", padding=3, fontsize=9)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    top = max(values) if values else 1.0
    ax.set_ylim(0, top * 1.25 + 1e-6)


def multi_run_grouped_bar(ax, runs, key_path, title, ylabel, fmt=".3f"):
    """
    Grouped bar chart comparing multiple runs for the same metric.
    Groups = model names; bars within each group = runs.
    Only model+run combos with non-None values are shown.
    """
    # Collect data
    run_labels = [r[0] for r in runs]
    data = {}  # run_label -> {model: value}
    for run_label, _, model_results in runs:
        data[run_label] = {
            m: _safe_get(model_results, m, *key_path)
            for m in MODEL_NAMES
        }

    # Only include models that have data in at least one run
    models = [m for m in MODEL_NAMES if any(data[r].get(m) is not None for r in run_labels)]
    if not models:
        ax.set_title(f"{title}\n(no data)")
        ax.axis("off")
        return

    n_runs  = len(run_labels)
    n_models = len(models)
    width   = 0.8 / n_runs
    x       = np.arange(n_models)
    run_colors = plt.cm.tab10(np.linspace(0, 0.6, n_runs))

    for i, (run_label, color) in enumerate(zip(run_labels, run_colors)):
        vals  = [data[run_label].get(m) for m in models]
        # Replace None with 0 for plotting but mask with hatch
        plot_vals = [v if v is not None else 0.0 for v in vals]
        hatches   = ['///' if v is None else '' for v in vals]
        offsets   = x - 0.4 + width * (i + 0.5)
        bars = ax.bar(offsets, plot_vals, width=width * 0.9, color=color,
                      label=run_label, edgecolor="white")
        for bar, hatch in zip(bars, hatches):
            bar.set_hatch(hatch)
        ax.bar_label(bars,
                     labels=[f"%{fmt}" % v if v is not None else "N/A" for v in vals],
                     padding=2, fontsize=8)

    ax.set_xticks(x)
    ax.set_xticklabels(models)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=9)
    all_vals = [v for d in data.values() for v in d.values() if v is not None]
    top = max(all_vals) if all_vals else 1.0
    ax.set_ylim(0, top * 1.3 + 1e-6)


def plot_metric(runs, key_path, title, ylabel, fmt=".3f", figsize=(7, 4)):
    """Top-level helper: single-run bar or multi-run grouped bar depending on len(runs)."""
    fig, ax = plt.subplots(figsize=figsize)
    if len(runs) == 1:
        single_run_bar(ax, runs[0][2], key_path, title, ylabel, fmt)
    else:
        multi_run_grouped_bar(ax, runs, key_path, title, ylabel, fmt)
    plt.tight_layout()
    plt.show()

## Instruction Following (Mean Helpfulness)

In [ ]:
plot_metric(
    runs,
    key_path=("instruction_following", "mean_helpfulness"),
    title="Instruction Following — Mean Helpfulness (GPT judge, 0–100)",
    ylabel="Score",
    fmt=".1f",
    figsize=(7, 4),
)

## Subliminal Effect Probes

Plots depend on `subliminal_type` stored in the result metadata.

In [ ]:
# ── preference_in_category: per-probe-type target frequency ───────────────────
if subliminal_type == "preference_in_category":
    probe_types = [
        ("probe_direct",          "Direct Probe"),
        ("probe_narrative",       "Narrative Probe"),
        ("probe_multiple_choice", "Multiple Choice Probe"),
    ]
    # Find probe types that have data in at least one run for at least one model
    active_probes = [
        (pt_key, pt_label)
        for pt_key, pt_label in probe_types
        if any(
            _safe_get(mr, m, "subliminal", pt_key, "target_frequency") is not None
            for _, _, mr in runs
            for m in MODEL_NAMES
        )
    ]
    if not active_probes:
        print("[preference_in_category] No probe data found — skipping.")
    else:
        ncols = len(active_probes)
        fig, axes = plt.subplots(1, ncols, figsize=(5 * ncols, 4), squeeze=False)
        for ax, (pt_key, pt_label) in zip(axes[0], active_probes):
            if len(runs) == 1:
                single_run_bar(
                    ax, runs[0][2],
                    ("subliminal", pt_key, "target_frequency"),
                    pt_label, "Target Word Frequency",
                )
            else:
                multi_run_grouped_bar(
                    ax, runs,
                    ("subliminal", pt_key, "target_frequency"),
                    pt_label, "Target Word Frequency",
                )
        fig.suptitle("Subliminal Preference Probe — Target Word Frequency", y=1.02, fontsize=13)
        plt.tight_layout()
        plt.show()

In [ ]:
# ── persona_behavior: misalignment rate + alignment score ─────────────────────
if subliminal_type == "persona_behavior":
    metrics = [
        (("subliminal", "misalignment_rate"), "Misalignment Rate",  "Rate",        ".3f"),
        (("subliminal", "mean_alignment"),    "Mean Alignment Score","Score (0–100)",".1f"),
        (("subliminal", "mean_coherence"),    "Mean Coherence Score","Score (0–100)",".1f"),
    ]
    active = [
        (kp, title, ylabel, fmt)
        for kp, title, ylabel, fmt in metrics
        if any(
            _safe_get(mr, m, *kp) is not None
            for _, _, mr in runs
            for m in MODEL_NAMES
        )
    ]
    if not active:
        print("[persona_behavior] No probe data found — skipping.")
    else:
        fig, axes = plt.subplots(1, len(active), figsize=(5 * len(active), 4), squeeze=False)
        for ax, (kp, title, ylabel, fmt) in zip(axes[0], active):
            if len(runs) == 1:
                single_run_bar(ax, runs[0][2], kp, title, ylabel, fmt)
            else:
                multi_run_grouped_bar(ax, runs, kp, title, ylabel, fmt)
        fig.suptitle("Subliminal Probe — Persona Behavior", y=1.02, fontsize=13)
        plt.tight_layout()
        plt.show()

In [ ]:
# ── language_insertion: target language rate ───────────────────────────────────
if subliminal_type == "language_insertion":
    kp = ("subliminal", "target_language_rate")
    has_data = any(
        _safe_get(mr, m, *kp) is not None
        for _, _, mr in runs
        for m in MODEL_NAMES
    )
    if not has_data:
        print("[language_insertion] No probe data found — skipping.")
    else:
        plot_metric(
            runs, kp,
            title="Subliminal Language Insertion — Target Language Rate",
            ylabel="Rate",
        )

In [ ]:
# ── code_security: misalignment + insecure rate ────────────────────────────────
if subliminal_type == "code_security":
    metrics = [
        (("subliminal", "misalignment_rate"), "Misalignment Rate",    "Rate",        ".3f"),
        (("subliminal", "mean_alignment"),    "Mean Alignment Score", "Score (0–100)",".1f"),
        (("subliminal", "insecure_rate"),      "Subliminal Insecure Code Rate", "Rate", ".3f"),
    ]
    active = [
        (kp, title, ylabel, fmt)
        for kp, title, ylabel, fmt in metrics
        if any(
            _safe_get(mr, m, *kp) is not None
            for _, _, mr in runs
            for m in MODEL_NAMES
        )
    ]
    if not active:
        print("[code_security] No subliminal probe data found — skipping.")
    else:
        fig, axes = plt.subplots(1, len(active), figsize=(5 * len(active), 4), squeeze=False)
        for ax, (kp, title, ylabel, fmt) in zip(axes[0], active):
            if len(runs) == 1:
                single_run_bar(ax, runs[0][2], kp, title, ylabel, fmt)
            else:
                multi_run_grouped_bar(ax, runs, kp, title, ylabel, fmt)
        fig.suptitle("Subliminal Probe — Code Security", y=1.02, fontsize=13)
        plt.tight_layout()
        plt.show()

## Coding Ability — Insecure Code Rate (desired-feature eval)

In [ ]:
kp = ("coding", "insecure_rate")
has_data = any(
    _safe_get(mr, m, *kp) is not None
    for _, _, mr in runs
    for m in MODEL_NAMES
)
if not has_data:
    print("[coding] No insecure_rate data found — skipping.")
else:
    plot_metric(
        runs, kp,
        title="Coding Ability — Insecure Code Rate (GPT judge)",
        ylabel="Insecure Rate",
    )